# 02 — Preprocess & align to the 1 km target grid

Reproject / resample / clip every input layer to the **shared analysis grid**
(ESRI:102008, 1 km, Y2Y boundary buffered by `BUFFER_KM`) and write a coregistered
raster stack to `input_data/cleaned_aligned/` — ready for `prioritizr`.

**This notebook reads and writes full pixel data** (unlike `01`, which is metadata-only)
and can run for a while. Ethan runs it; Claude never executes cells.

**Engine:** system **`gdalwarp`** — one streamed reproject + resample + cutline-clip per
layer. It reads only the Y2Y window of each source, so global rasters (e.g. AOH ~4 GB)
are never warped in full. Config (paths, per-layer `resampling`, `build_vrt`, `multi`)
comes from `config.py`.

**Steps:** build study area + grid → rebuild gHM VRT → align single-file layers →
align every EFG that occurs in Y2Y → verify the stack coregisters.

**Kernel:** select **`Python (y2y-geo)`** (the project `.venv`, Python 3.12).

In [ ]:
import importlib
import math
import subprocess
import shutil
import time

import numpy as np
import pyproj
import rasterio
import geopandas as gpd

import config
importlib.reload(config)  # pick up edits to config.py when re-running this cell
from config import (
    DATASETS, INPUT_DIR, ALIGNED_DIR, CORRIDOR_REF,
    TARGET_CRS, TARGET_RES_M, BUFFER_KM,
    find_rasters, pick_representative, study_area,
)

# This notebook drives the system GDAL CLIs (osgeo bindings aren't in the venv).
for cli in ("gdalbuildvrt", "gdalwarp"):
    assert shutil.which(cli), f"{cli} not found on PATH (need system GDAL CLIs)"
print("GDAL CLIs OK | rasterio", rasterio.__version__, "| geopandas", gpd.__version__)
print("datasets in config:", len(DATASETS))

In [ ]:
# ---- Study area + 1 km target grid ---------------------------------------
ALIGNED_DIR.mkdir(parents=True, exist_ok=True)

sa = study_area(BUFFER_KM)  # Y2Y boundary, ESRI:102008, buffered by BUFFER_KM
res = TARGET_RES_M
minx, miny, maxx, maxy = sa.total_bounds
left = math.floor(minx / res) * res  # snap extent to a clean 1 km grid
bottom = math.floor(miny / res) * res
right = math.ceil(maxx / res) * res
top = math.ceil(maxy / res) * res
TE = [left, bottom, right, top]  # target extent, in TARGET_CRS (gdalwarp -te)
WIDTH = int(round((right - left) / res))
HEIGHT = int(round((top - bottom) / res))
TRANSFORM = rasterio.transform.from_origin(left, top, res, res)

# Cutline used to clip every layer to the buffered corridor.
CUTLINE_PATH = ALIGNED_DIR / "_study_area.gpkg"
sa.to_file(CUTLINE_PATH, driver="GPKG")

print(f"target grid : {WIDTH} x {HEIGHT} cells @ {res} m in {TARGET_CRS}")
print(f"extent (-te): {TE}")
print(f"cutline     -> {CUTLINE_PATH.relative_to(INPUT_DIR.parent)}")

In [ ]:
# ---- Rebuild the gHM mosaic VRT from its tiles (recorded in-workflow) -----
# The original VRT was deleted; we rebuild it here so the mosaic is reproducible.
hm = DATASETS["human_modification"]
hm_tiles = find_rasters(hm)  # the 4 gHM GeoTIFF tiles
HM_VRT = hm["path"] / "HM_Y2Y_2024_90_60land.vrt"
proc = subprocess.run(
    ["gdalbuildvrt", str(HM_VRT), *map(str, hm_tiles)],
    capture_output=True, text=True,
)
if proc.returncode != 0:
    raise RuntimeError(f"gdalbuildvrt failed:\n{proc.stderr}")
print(f"built VRT from {len(hm_tiles)} tiles -> {HM_VRT.name}")

In [ ]:
# ---- Alignment helpers ---------------------------------------------------
GDAL_RESAMP = {"average": "average", "bilinear": "bilinear", "nearest": "near"}


def pick_nodata(src):
    """Output NoData: keep the source's; else NaN (float) / 0 (integer)."""
    if src.nodata is not None:
        return src.nodata
    return float("nan") if np.issubdtype(np.dtype(src.dtypes[0]), np.floating) else 0


def source_for(name, cfg):
    """The raster to align: the rebuilt VRT for gHM, else the representative."""
    if cfg.get("build_vrt"):
        return HM_VRT
    return pick_representative(cfg, find_rasters(cfg))


def warp_to_grid(src, dst, resampling, nodata):
    """Reproject + resample + clip a raster to the shared 1 km grid via gdalwarp.
    gdalwarp reads only the source window covering the target extent, so global
    rasters are never warped in full (clip-before-reproject is implicit). The
    cutline masks everything outside the buffered corridor to NoData while the
    fixed -te/-tr keeps every output on the identical grid."""
    dst.parent.mkdir(parents=True, exist_ok=True)
    cmd = [
        "gdalwarp", "-overwrite",
        "-t_srs", TARGET_CRS,
        "-te", *map(str, TE),
        "-tr", str(TARGET_RES_M), str(TARGET_RES_M),
        "-r", GDAL_RESAMP[resampling],
        "-cutline", str(CUTLINE_PATH),
        "-dstnodata", str(nodata),
        "-of", "GTiff",
        "-co", "TILED=YES", "-co", "COMPRESS=DEFLATE", "-co", "BIGTIFF=IF_SAFER",
        "-multi", "-wo", "NUM_THREADS=ALL_CPUS",
        str(src), str(dst),
    ]
    proc = subprocess.run(cmd, capture_output=True, text=True)
    if proc.returncode != 0:
        raise RuntimeError(f"gdalwarp failed for {src}:\n{proc.stderr}")


def align_dataset(name, cfg, dst):
    """Warp one dataset's source raster onto the shared grid."""
    src = source_for(name, cfg)
    with rasterio.open(src) as s:
        nd = pick_nodata(s)
    warp_to_grid(src, dst, cfg["resampling"], nd)
    return dst

In [ ]:
# ---- Align all single-file datasets --------------------------------------
single = {n: c for n, c in DATASETS.items() if not c.get("multi")}
for name, cfg in single.items():
    t0 = time.perf_counter()
    dst = ALIGNED_DIR / f"{name}.tif"
    align_dataset(name, cfg, dst)
    print(f"[{name:30s}] {cfg['resampling']:8s} -> {dst.name}  ({time.perf_counter() - t0:5.1f}s)")
print("single-file datasets aligned.")

In [ ]:
# ---- IUCN EFGs: align every grid that actually occurs in the study area ---
# EFG maps are continental/global in extent, so a bbox test always "overlaps".
# Real overlap = the EFG has present cells (value > 0) inside the corridor after
# clipping. We warp each (cheap -- they're coarse), then drop any with no presence.
# NOTE: assumes EFG values encode occurrence as > 0 (0/nodata = absent); verify the
# value scheme of the GET maps if the kept/dropped counts look off.
efg = DATASETS["iucn_efg"]
efg_rasters = find_rasters(efg)
efg_out = ALIGNED_DIR / "iucn_efg"
kept, dropped = [], []
for i, src in enumerate(efg_rasters, 1):
    dst = efg_out / f"{src.stem}.tif"
    with rasterio.open(src) as s:
        nd = pick_nodata(s)
    warp_to_grid(src, dst, efg["resampling"], nd)
    with rasterio.open(dst) as s:
        arr = s.read(1, masked=True)
    m = np.ma.getmaskarray(arr)
    present = int(((arr.data > 0) & ~m).sum())
    if present == 0:
        dst.unlink()  # no occurrence in Y2Y -> not a feature for this study area
        dropped.append(src.stem)
    else:
        kept.append((src.stem, present))
    if i % 20 == 0:
        print(f"  ...processed {i}/{len(efg_rasters)} EFGs")

print(f"EFGs: kept {len(kept)} / {len(efg_rasters)} (dropped {len(dropped)} with no Y2Y presence)")
print("dropped:", ", ".join(dropped) if dropped else "(none)")

In [ ]:
# ---- Verify the stack coregisters ----------------------------------------
# The core correctness check for a prioritizr-ready stack: every output must
# share the exact same CRS, transform, and shape.
def _same(t1, t2):
    return all(abs(a - b) < 1e-6 for a, b in zip(t1, t2))


target_crs_obj = pyproj.CRS.from_user_input(TARGET_CRS)
single_outs = sorted(ALIGNED_DIR.glob("*.tif"))
efg_outs = sorted((ALIGNED_DIR / "iucn_efg").glob("*.tif"))
ref_t = tuple(TRANSFORM)[:6]
bad = []
for p in single_outs + efg_outs:
    with rasterio.open(p) as s:
        ok = (
            s.width == WIDTH
            and s.height == HEIGHT
            and _same(tuple(s.transform)[:6], ref_t)
            and s.crs is not None
            and pyproj.CRS.from_user_input(s.crs) == target_crs_obj
        )
    if not ok:
        bad.append(p.name)

print(f"aligned layers: {len(single_outs)} single + {len(efg_outs)} EFG = {len(single_outs) + len(efg_outs)}")
print(f"grid: {WIDTH} x {HEIGHT} @ {TARGET_RES_M} m, {TARGET_CRS}")
assert not bad, f"GRID MISMATCH in: {bad}"
print("OK: every layer shares CRS / transform / shape -> stack is coregistered.")